<a href="https://colab.research.google.com/github/chiraswykj/GENAI_TUTORIALS/blob/main/GENAI4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q pymupdf sentence-transformers transformers faiss-cpu

In [ ]:
import pymupdf
import numpy as np

from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
uploaded = files.upload()

pdf_path = next(iter(uploaded))

print("PDF uploaded:", pdf_path)

Saving OOP_notes.pdf to OOP_notes.pdf
PDF uploaded: OOP_notes.pdf


In [ ]:
doc = pymupdf.open(pdf_path)

text = ""

for page in doc:
    text += page.get_text() + "\n"

doc.close()

print("Text extracted successfully!")
print(text[:1000])

Text extracted successfully!


























































In [4]:
if 'text' not in locals():
    print("Error: The 'text' variable is not defined. Please make sure to run the cell above (wKGXOvRiWeqL) to extract the text from your uploaded PDF first!")
else:
    print(f"Extracted text length: {len(text)}")
    if len(text.strip()) == 0:
        print("Warning: Extracted text is empty. Checking PDF structure...")
    else:
        print(f"Sample text: {text[:200]!r}")

    chunk_size = 800

    # Safely extract chunks of raw text
    chunks = [
        text[i:i + chunk_size]
        for i in range(0, len(text), chunk_size)
        if text[i:i + chunk_size].strip()
    ]

    print("Number of chunks:", len(chunks))

Error: The 'text' variable is not defined. Please make sure to run the cell above (wKGXOvRiWeqL) to extract the text from your uploaded PDF first!


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Since chunks is a list of plain strings, we should embed them directly
if len(chunks) > 0:
    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True
    ).astype("float32")
    print("Embedding shape:", embeddings.shape)
else:
    embeddings = np.empty((0, 384), dtype="float32")
    print("No chunks to embed. Created empty embeddings array.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (0,)


In [2]:
import faiss
import numpy as np

try:
    if 'embeddings' in locals() and isinstance(embeddings, np.ndarray) and embeddings.ndim == 2 and embeddings.shape[0] > 0:
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embeddings)
        print("Vectors stored successfully in FAISS:", index.ntotal)
    else:
        print("Cannot build FAISS index: Embeddings array is empty. Please verify why no chunks were created from your PDF.")
except Exception as e:
    print(f"Error initializing FAISS: {e}")

ModuleNotFoundError: No module named 'faiss'

In [8]:
# Consolidated Pipeline to run everything end-to-end safely
import pymupdf
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import os
import re
from google.colab import files

try:
    # Automatically find pdf_path, upload if missing
    if 'pdf_path' not in locals():
        if os.path.exists('OOP_notes.pdf'):
            pdf_path = 'OOP_notes.pdf'
            print("Found 'OOP_notes.pdf' locally. Continuing pipeline...")
        else:
            print("--- Step 0: Upload PDF ---")
            print("PDF file not found in current session. Please upload your PDF (e.g., OOP_notes.pdf) now:")
            uploaded = files.upload()
            pdf_path = next(iter(uploaded))
            print(f"Uploaded successfully: {pdf_path}")

    # 1. Extract Text
    print("\n--- Step 1: Extracting Text ---")
    doc = pymupdf.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text() + "\n"
    doc.close()

    # Check if the extracted text contains actual alphanumeric content
    alphanumeric_chars = len(re.sub(r'[^a-zA-Z0-9]', '', text))
    print(f"Raw text length: {len(text)}")
    print(f"Alphanumeric character count: {alphanumeric_chars}")

    if alphanumeric_chars == 0:
        print("\n[ERROR / WARNING]: The PDF was read, but no readable text or letters were found.")
        print("This usually means the PDF consists of scanned images. Please upload an OCR-processed (searchable) version of the PDF.")

    # 2. Chunk Text
    print("\n--- Step 2: Chunking Text ---")
    chunk_size = 800
    chunks = [
        text[i:i + chunk_size]
        for i in range(0, len(text), chunk_size)
        if text[i:i + chunk_size].strip()
    ]
    print(f"Success! Created {len(chunks)} chunks.")

    # 3. Create Embeddings
    if len(chunks) > 0:
        print("\n--- Step 3: Generating Embeddings ---")
        embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
        embeddings = embedding_model.encode(chunks, convert_to_numpy=True).astype("float32")
        print(f"Success! Generated embeddings with shape: {embeddings.shape}")

        # 4. Build FAISS Index
        print("\n--- Step 4: Storing in FAISS ---")
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embeddings)
        print(f"Success! FAISS Index populated with {index.ntotal} vectors.")
    else:
        print("\nAborted: No valid chunks found to generate embeddings.")

except Exception as e:
    print(f"An error occurred during execution: {e}")


--- Step 1: Extracting Text ---
Raw text length: 55
Alphanumeric character count: 0

[ERROR / WARNING]: The PDF was read, but no readable text or letters were found.
This usually means the PDF consists of scanned images. Please upload an OCR-processed (searchable) version of the PDF.

--- Step 2: Chunking Text ---
Success! Created 0 chunks.

Aborted: No valid chunks found to generate embeddings.
